In [0]:
CATALOG="workspace"
SCHEMA="incident_assistent"
VOLUME="docs"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
base = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print(base)

/Volumes/workspace/incident_assistent/docs


In [0]:
import json, os

docs = []
for fn in ["docs_part1.json", "docs_part2.json"]:
    with open(f"{base}/{fn}") as f:
        docs += json.load(f)

os.makedirs(f"{base}/runbooks", exist_ok=True)
for d in docs:
    with open(f"{base}/runbooks/{d['doc_id']}.md", "w") as out:
        out.write(d["title"] + "\n\n" + d["text"])

print(len(docs), "runbook files written")

30 runbook files written


In [0]:
display(dbutils.fs.ls(f"{base}/runbooks"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/adf_pipeline_failed_activity.md,adf_pipeline_failed_activity.md,912,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/adf_trigger_missed.md,adf_trigger_missed.md,824,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/backfill_procedure.md,backfill_procedure.md,938,1792000039000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/bad_csv_delimiter.md,bad_csv_delimiter.md,921,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/checkpoint_corruption.md,checkpoint_corruption.md,861,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/compute_start_failure.md,compute_start_failure.md,933,1792000037000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/concurrent_write_error.md,concurrent_write_error.md,866,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/corrupt_json_record.md,corrupt_json_record.md,827,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/data_quality_check_failing.md,data_quality_check_failing.md,955,1792000038000
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/data_skew.md,data_skew.md,1080,1792000037000


In [0]:
from pyspark.sql import functions as F

bronze_table = f"{CATALOG}.{SCHEMA}.bronze_docs"

raw = (spark.read.format("text").option("wholetext", "true")
       .load(f"{base}/runbooks/*.md")
       .select(F.col("_metadata.file_path").alias("file_path"),
               F.col("value").alias("text")))

raw = (raw.withColumn("doc_id", F.regexp_extract("file_path", r"([^/]+)\.md$", 1))
          .withColumn("doc_hash", F.md5("text")))

raw.write.mode("overwrite").saveAsTable(bronze_table)
display(spark.table(bronze_table))

file_path,text,doc_id,doc_hash
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/adf_pipeline_failed_activity.md,"Azure Data Factory Pipeline Failure Due to Activity Error **Symptoms:** ADF monitoring dashboard reports a pipeline failure status, with the Databricks notebook activity showing a red cross and failure exit code. **Root Cause:** The underlying PySpark notebook encountered an unhandled exception (such as a bad file path, syntax error, or assertion failure) and exited with a non-zero status code. **Step-by-Step Fix:** 1. Click on the failed ADF activity run and follow the link to the Databricks cluster driver logs and notebook output. 2. Inspect the stack trace to identify the exact line of code causing the failure. 3. Fix the notebook code, commit changes to git, and rerun the failed ADF pipeline from the failed activity. **Prevention:** Implement robust try-except error handling blocks within PySpark scripts and add custom logging to capture actionable error messages before notebook termination.",adf_pipeline_failed_activity,5e17c5da61aadfd4e26d9a7545c8dbb0
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/adf_trigger_missed.md,"Azure Data Factory Scheduled Trigger Missed Execution **Symptoms:** Expected daily ETL pipelines fail to kick off at their scheduled time, resulting in stale downstream analytical dashboards. **Root Cause:** The ADF tumbling window or schedule trigger was accidentally left in a `Stopped` state, or encountered transient platform throttling during scheduling evaluation. **Step-by-Step Fix:** 1. Navigate to the ADF Author/Monitor hub and inspect the trigger status. 2. Start the stopped trigger manually through the portal or Azure PowerShell SDK. 3. Use the ADF rerun functionality to backfill missing window runs for the affected dates. **Prevention:** Set up Azure Monitor alerts for trigger failures and stopped states. Use CI/CD deployment pipelines to ensure triggers are automatically published in active states.",adf_trigger_missed,fdee35c336f699ba856dd7aa92d2f8fe
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/backfill_procedure.md,"Historical Backfill Procedure Overwriting Current Data **Symptoms:** Running a historical backfill pipeline inadvertently wipes out or corrupts the most recent production data partitions in the silver Delta table. **Root Cause:** The backfill script was executed without restricting date filters or used static partition overwrite mode, causing it to overwrite the entire table root instead of targeted historical date ranges. **Step-by-Step Fix:** 1. Access Delta table history using `DESCRIBE HISTORY table_name` to locate the version prior to the backfill execution. 2. Restore the table state using `RESTORE TABLE table_name TO VERSION AS OF N`. 3. Rerun the backfill job with strict date filter predicates and dynamic partition overwrite enabled. **Prevention:** Design all backfill notebooks to accept parameterized start and end dates. Test backfill scripts in development environments before running against production tables.",backfill_procedure,85506b9751170dbd4378c72f2cfd25d4
dbfs:/Volumes/workspace/incident_assistent/docs/runbooks/bad_csv_delimiter.md,"Bad CSV Delimiter Causing Column Shifting and Parsing Errors **Symptoms:** Downstream Delta tables contain scrambled data columns, misplaced values, or rows shifted by several columns after ingesting flat CSV files. **Root Cause:** The upstream CSV file contained unescaped embedded commas or quotation marks within text fields, while the Spark CSV reader was configured with default comma delimiters and standard quote settings. **Step-by-Step Fix:** 1. Inspect raw CSV files in ADLS Gen2 using a text editor to check delimiter and quote character usage. 2. Update Spark read options explicitly: `.option('delimiter', ',').option('quote', '""').option('escape', '""')`. 3. Reprocess the affected partition by overwriting the corrupted bronze and silver layers. **Prevention:** Enforce strict file format s